# Week 57, Day 3: Maximum-margin classifiers

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w057/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Solving for the widest margin

Solve the hard-margin SVM with scipy's SLSQP: the variable is v = (w1, w2, b), the objective 0.5 (w1² + w2²), and one inequality constraint y_i (w · x_i + b) - 1 >= 0 per example. Report the margin and the support vectors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(573)
Y = np.repeat([-1.0, 1.0], 20)
Xs = 0.5 * r.normal(size=(40, 2)) + np.outer(Y, [1.1, 0.6])

cons = {"type": "ineq", "fun": lambda v: ...}
res = minimize(lambda v: 0.5 * (v[0] ** 2 + v[1] ** 2), x0=np.array([1.0, 1.0, 0.0]), constraints=[cons], method="SLSQP")
w_svm, b_svm = res.x[:2], res.x[2]
margin = 1 / np.linalg.norm(w_svm)
sv = np.flatnonzero(np.abs(Y * (Xs @ w_svm + b_svm) - 1) < 1e-3)
print(f"w = {np.round(w_svm, 3)}, b = {b_svm:.3f}; margin {margin:.3f}; support vectors {[int(i) for i in sv]}")
xx = np.linspace(-2.5, 2.5, 2)
for c, ls in [(0, "-"), (1, "--"), (-1, "--")]:
    plt.plot(xx, (c - b_svm - w_svm[0] * xx) / w_svm[1], "k" + ls)
plt.scatter(Xs[:, 0], Xs[:, 1], c=Y, cmap="coolwarm"); plt.scatter(Xs[sv, 0], Xs[sv, 1], s=120, facecolors="none", edgecolors="k"); plt.show()

In [ ]:
import numpy as np
def test_feasible():
    "every training example satisfies y (w · x + b) >= 1 at the solution"
    assert np.all(Y * (Xs @ w_svm + b_svm) >= 1 - 1e-4)
def test_sv():
    "there are between 2 and 6 support vectors, and they have y f(x) = 1"
    assert 2 <= len(sv) <= 6 and np.allclose(Y[sv] * (Xs[sv] @ w_svm + b_svm), 1, atol=1e-3)
def test_margin():
    "the margin is 1 / |w|"
    assert abs(margin - 1 / np.linalg.norm(w_svm)) < 1e-12

test_feasible()
test_sv()
test_margin()
print("All checks passed")

### Exercise: A soft-margin SVM by subgradient descent

Write hinge(m). Then run Pegasos on overlapping classes: at step t with learning rate 1 / (lam t), pick a random example; if its margin y f(x) is below 1, step w towards y x, and always shrink w by (1 - lr lam). Update b without shrinkage. Report test accuracy.

In [ ]:
def hinge(m):
    return ...

Yo = np.repeat([-1.0, 1.0], 200); Xo = 1.0 * r.normal(size=(400, 2)) + np.outer(Yo, [0.8, 0.5])
Yot = np.repeat([-1.0, 1.0], 2000); Xot = 1.0 * r.normal(size=(4000, 2)) + np.outer(Yot, [0.8, 0.5])
lam = 0.01
w_soft, b_soft = np.zeros(2), 0.0
for t in range(1, 20001):
    i = r.integers(400); lr = 1 / (lam * t)
    m = Yo[i] * (Xo[i] @ w_soft + b_soft)
    w_soft *= (1 - lr * lam)
    if m < 1:
        w_soft += ...
        b_soft += 0.1 * lr * lam * Yo[i]
acc_svm = np.mean(np.sign(Xot @ w_soft + b_soft) == Yot)
print(f"w = {np.round(w_soft, 3)}, b = {b_soft:.3f}; test accuracy {acc_svm:.3f}")
print(f"mean hinge loss on training data {hinge(Yo * (Xo @ w_soft + b_soft)).mean():.3f}")
mm = np.linspace(-3, 3, 200)
plt.plot(mm, hinge(mm), label="hinge"); plt.plot(mm, np.log1p(np.exp(-mm)) / np.log(2), label="logistic (scaled)"); plt.plot(mm, (mm < 0).astype(float), label="0-1")
plt.xlabel("margin y f(x)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_hinge():
    "hinge(m) is max(0, 1 - m)"
    assert np.allclose(hinge(np.array([-1.0, 0.5, 1.0, 2.0])), [2.0, 0.5, 0.0, 0.0])
def test_fit():
    "the soft-margin SVM reaches test accuracy above 0.8 and most examples lie outside the margin"
    assert acc_svm > 0.8 and np.mean(Yo * (Xo @ w_soft + b_soft) >= 1) > 0.5

test_hinge()
test_fit()
print("All checks passed")

### Exercise: Gradient descent finds the maximum margin

Run gradient descent on the unpenalised logistic loss for the separable data (labels as 0 and 1, with an intercept column), recording at several checkpoints the angle in degrees between the weight vector (without the intercept) and w_svm.

In [ ]:
A = np.c_[np.ones(40), Xs]; t01 = (Y + 1) / 2
w = np.zeros(3); angles, norms = [], []; checkpoints = [100, 1000, 10000, 50000]
for step in range(1, 50001):
    p = 1 / (1 + np.exp(-A @ w))
    w -= 0.5 * A.T @ (p - t01) / 40
    if step in checkpoints:
        cosang = ...
        angles.append(np.degrees(np.arccos(np.clip(cosang, -1, 1)))); norms.append(np.linalg.norm(w[1:]))
for c, a, nw in zip(checkpoints, angles, norms):
    print(f"after {c:6d} steps: |w| = {nw:.1f}; angle to the SVM direction {a:.2f} degrees")

In [ ]:
import numpy as np
def test_angle():
    "the angle between the logistic direction and the SVM direction shrinks with training and ends below 5 degrees"
    assert angles[-1] < angles[0] and angles[-1] < 5

test_angle()
print("All checks passed")